# Ablation: Distance / Similarity Metrics

Tests how the choice of similarity metric affects retrieval quality and speed:

**Dense retrieval:**
- Cosine similarity (L2-normalize embeddings + inner product) — current default
- Dot product (inner product without normalization)
- L2 / Euclidean distance (IndexFlatL2)

**BM25:**
- Vary k1 (term saturation) ∈ {0.5, 1.2, 1.5, 2.0}
- Vary b (length normalization) ∈ {0.0, 0.5, 0.75, 1.0}


In [1]:
import sys, os, gc
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import faiss

from loader import load_data
from retrievers.bm25 import BM25Retriever
from encoders.sbert import SentenceBERTEncoder
from evaluation.mrr import mrr_at_10
from evaluation.timing import measure_retrieval_time

N = 50_000
TOP_K = 10

ds = load_data(n=N)
passages_text, queries = [], []
for example in ds:
    queries.append(example["query"])
    for p in example["passages"]["passage_text"]:
        passages_text.append(p)
print(f"Passages: {len(passages_text)}, Queries: {len(queries)}")

# Load or generate SBERT embeddings once (un-normalized)
EMB_FILE = "sbert_embeddings.npy"
encoder = SentenceBERTEncoder()
if os.path.exists(EMB_FILE) and np.load(EMB_FILE, mmap_mode='r').shape[0] == len(passages_text):
    raw_embs = np.load(EMB_FILE).astype(np.float32)
else:
    raw_embs = encoder.encode(passages_text).astype(np.float32)
    np.save(EMB_FILE, raw_embs)
print(f"Embeddings shape: {raw_embs.shape}")

/home/skimura/Projects/ics624_document_retrieval_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Passages: 498725, Queries: 50000


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12305.62it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SentenceBERTEncoder using device: cuda
Embeddings shape: (498725, 384)


## 1. Dense Retrieval — Similarity Metrics

In [2]:
def make_dense_retriever(index, embs, normalize_queries):
    """Generic dense retriever using any FAISS index and query normalization flag."""
    class _Retriever:
        def query(self, q):
            q_emb = encoder.encode([q]).astype(np.float32)
            if normalize_queries:
                q_emb /= np.linalg.norm(q_emb, axis=1, keepdims=True)
            _, idx = index.search(q_emb, TOP_K)
            return idx[0].tolist()

        def query_batch(self, qs):
            q_embs = encoder.encode(qs).astype(np.float32)
            if normalize_queries:
                q_embs /= np.linalg.norm(q_embs, axis=1, keepdims=True)
            _, idxs = index.search(q_embs, TOP_K)
            return idxs.tolist()
    return _Retriever()

In [3]:
results = []

# --- Cosine similarity (L2-normalize + IndexFlatIP) ---
normed = raw_embs / np.linalg.norm(raw_embs, axis=1, keepdims=True)
idx_cos = faiss.IndexFlatIP(normed.shape[1])
idx_cos.add(normed)
r = make_dense_retriever(idx_cos, normed, normalize_queries=True)
mrr = mrr_at_10(r, ds)
t   = measure_retrieval_time(r, queries)
results.append({"Method": "Dense", "Metric": "Cosine (normalize + IP)", "MRR@10": round(mrr, 4), "Avg ms/query": round(t*1000, 3)})
print(f"Cosine   MRR: {mrr:.4f}, {t*1000:.1f} ms/q")
del idx_cos, normed; gc.collect()

# --- Dot product (IndexFlatIP, no normalization) ---
idx_ip = faiss.IndexFlatIP(raw_embs.shape[1])
idx_ip.add(raw_embs)
r = make_dense_retriever(idx_ip, raw_embs, normalize_queries=False)
mrr = mrr_at_10(r, ds)
t   = measure_retrieval_time(r, queries)
results.append({"Method": "Dense", "Metric": "Dot Product (no normalize)", "MRR@10": round(mrr, 4), "Avg ms/query": round(t*1000, 3)})
print(f"Dot Prod MRR: {mrr:.4f}, {t*1000:.1f} ms/q")
del idx_ip; gc.collect()

# --- L2 / Euclidean distance (IndexFlatL2) ---
# Lower L2 = more similar, so FAISS returns nearest by L2; closer = better
idx_l2 = faiss.IndexFlatL2(raw_embs.shape[1])
idx_l2.add(raw_embs)
r = make_dense_retriever(idx_l2, raw_embs, normalize_queries=False)
mrr = mrr_at_10(r, ds)
t   = measure_retrieval_time(r, queries)
results.append({"Method": "Dense", "Metric": "L2 / Euclidean", "MRR@10": round(mrr, 4), "Avg ms/query": round(t*1000, 3)})
print(f"L2       MRR: {mrr:.4f}, {t*1000:.1f} ms/q")
del idx_l2; gc.collect()

Running query_batch on 100 queries...


Batches: 100%|██████████| 1/1 [00:00<00:00, 300.69it/s]


Cosine   MRR: 0.5363, 40.9 ms/q
Running query_batch on 100 queries...


Batches: 100%|██████████| 1/1 [00:00<00:00, 292.08it/s]


Dot Prod MRR: 0.5363, 41.4 ms/q
Running query_batch on 100 queries...


Batches: 100%|██████████| 1/1 [00:00<00:00, 314.32it/s]


L2       MRR: 0.5363, 42.9 ms/q


12

## 2. BM25 — k1 and b Parameters

`k1` controls term frequency saturation (higher = more reward for repeated terms).
`b` controls document length normalization (0 = no normalization, 1 = full normalization).

In [4]:
# Sweep k1 (fix b=0.75)
for k1 in [0.5, 1.2, 1.5, 2.0]:
    r = BM25Retriever(top_k=TOP_K, k1=k1, b=0.75)
    r.fit(passages_text)
    mrr = mrr_at_10(r, ds)
    t   = measure_retrieval_time(r, queries)
    results.append({"Method": "BM25", "Metric": f"k1={k1}, b=0.75", "MRR@10": round(mrr, 4), "Avg ms/query": round(t*1000, 3)})
    print(f"BM25 k1={k1}, b=0.75  MRR: {mrr:.4f}, {t*1000:.1f} ms/q")
    del r; gc.collect()

Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100
BM25 k1=0.5, b=0.75  MRR: 0.3699, 132.8 ms/q
Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100
BM25 k1=1.2, b=0.75  MRR: 0.3485, 135.1 ms/q
Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100
BM25 k1=1.5, b=0.75  MRR: 0.3439, 135.7 ms/q
Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100
BM25 k1=2.0, b=0.75  MRR: 0.3353, 134.9 ms/q


In [5]:
# Sweep b (fix k1=1.5)
for b in [0.0, 0.5, 0.75, 1.0]:
    r = BM25Retriever(top_k=TOP_K, k1=1.5, b=b)
    r.fit(passages_text)
    mrr = mrr_at_10(r, ds)
    t   = measure_retrieval_time(r, queries)
    results.append({"Method": "BM25", "Metric": f"k1=1.5, b={b}", "MRR@10": round(mrr, 4), "Avg ms/query": round(t*1000, 3)})
    print(f"BM25 k1=1.5, b={b}    MRR: {mrr:.4f}, {t*1000:.1f} ms/q")
    del r; gc.collect()

Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100
BM25 k1=1.5, b=0.0    MRR: 0.2979, 139.5 ms/q
Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100
BM25 k1=1.5, b=0.5    MRR: 0.3415, 134.0 ms/q
Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100
BM25 k1=1.5, b=0.75    MRR: 0.3439, 134.6 ms/q
Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100
BM25 k1=1.5, b=1.0    MRR: 0.3339, 141.4 ms/q


## Results

In [6]:
df = pd.DataFrame(results)
df.to_csv("ablation_distance_metrics.csv", index=False)
print(df.to_string(index=False))

Method                     Metric  MRR@10  Avg ms/query
 Dense    Cosine (normalize + IP)  0.5363        40.908
 Dense Dot Product (no normalize)  0.5363        41.415
 Dense             L2 / Euclidean  0.5363        42.917
  BM25             k1=0.5, b=0.75  0.3699       132.833
  BM25             k1=1.2, b=0.75  0.3485       135.137
  BM25             k1=1.5, b=0.75  0.3439       135.675
  BM25             k1=2.0, b=0.75  0.3353       134.894
  BM25              k1=1.5, b=0.0  0.2979       139.514
  BM25              k1=1.5, b=0.5  0.3415       133.986
  BM25             k1=1.5, b=0.75  0.3439       134.607
  BM25              k1=1.5, b=1.0  0.3339       141.391
